In [1]:
# -*- coding: utf-8 -*-
"""
TEP-VOTE  |  nested, leakage-free, VST-normalised, nested-k version
==================================================================
Inside EVERY training split (5 CV folds + the final full-training fit):
  1. zero filter   : drop genes that are zero in >50 % of training samples
  2. size factors  : median-of-ratios, geometric means from TRAINING samples only
  3. PyDESeq2 on RAW counts of the training split -> DEGs (padj < 0.05, |log2FC| > 0.8)
     and the parametric dispersion trend (a0, a1) of that same training split
  4. VST (DESeq2 closed form, parametric trend)  -> applied to val/test with TRAIN parameters
  5. variance filter (var >= 0.1 on VST values), ANOVA F ranking, StandardScaler
  6. SMOTE / class weights -> 9-model soft vote

FINAL MODEL : k sweep on the full training set (pooled OOF AUC) -> best k -> ONE evaluation on held-out test.
HONEST CV   : k is selected AGAIN inside every outer training fold (inner CV), the outer fold is scored
              with that k, and the Youden threshold of each outer fold comes from the OTHER outer folds.
              The k-selection step is therefore part of what the CV estimate measures.
"""
from __future__ import annotations

import importlib.util
import os
import subprocess
import sys


def ensure(pkgs):
    for mod, pip_name in pkgs:
        if importlib.util.find_spec(mod) is None:
            print(f"Installing {pip_name} ...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])


ensure([("numpy", "numpy"), ("pandas", "pandas"), ("scipy", "scipy"), ("sklearn", "scikit-learn"),
        ("imblearn", "imbalanced-learn"), ("xgboost", "xgboost"), ("lightgbm", "lightgbm"),
        ("catboost", "catboost"), ("pydeseq2", "pydeseq2"), ("shap", "shap"),
        ("seaborn", "seaborn"), ("matplotlib", "matplotlib"), ("joblib", "joblib"),
        ("tqdm", "tqdm"), ("mygene", "mygene"), ("torch", "torch")])

os.environ["SCIPY_ARRAY_API"] = "1"   # before sklearn / scipy imports

import itertools
import logging
import pickle
import platform
import random
import time
import warnings
from dataclasses import dataclass

import joblib
import lightgbm as lgb
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
from scipy import stats
from scipy.stats import spearmanr
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_selection import f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, confusion_matrix, precision_recall_curve,
                             roc_auc_score, roc_curve)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
from xgboost import XGBClassifier

try:
    import shap
    SHAP_OK = True
except Exception:
    SHAP_OK = False

matplotlib.use("Agg")                        # figures are saved to disk, not shown
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


# ══════════════════════════════════════════════════════════════════════════
# [1] CONFIG
# ══════════════════════════════════════════════════════════════════════════
@dataclass
class Config:
    seed: int = 42
    n_splits: int = 5

    # preprocessing / DEG
    zero_frac: float = 0.50      # drop genes zero in > 50 % of training samples
    var_min: float = 0.10        # variance filter on VST values
    deg_padj: float = 0.05
    deg_lfc: float = 0.8         # |log2FC| > 0.8  (state this cutoff in the paper's Methods)
    deg_max: int | None = None   # None = no cap on DEG pool

    # k sweep (final model)
    sfs_start: int = 170
    sfs_step: int = 1
    sfs_max: int = 250
    best_k_override: int | None = None   # e.g. 197 to force a value, None = best pooled OOF AUC
    smote_k: int = 3
    resume: bool = True          # reuse cache files when settings are unchanged

    # nested-k CV (honest CV estimate: k selected inside every outer training fold)
    inner_splits: int = 3
    inner_k_stride: int = 10                      # inner search uses every 10th candidate k
    inner_models: tuple = ("SVM", "LR", "LGBM", "ET")   # fast proxy ensemble for the inner k search
    inner_full: bool = False     # True: rebuild DESeq2/ANOVA inside every inner split (15 extra DESeq2 fits)

    # models
    svm_C: float = 1.0
    svm_gamma: str = "scale"
    xgb_n_est: int = 300; xgb_depth: int = 4; xgb_lr: float = 0.05; xgb_early_stop: int = 20
    lgbm_n_est: int = 300; lgbm_depth: int = 4; lgbm_lr: float = 0.05; lgbm_early_stop: int = 20
    lr_C: float = 1.0; lr_max_iter: int = 2000
    rf_n_est: int = 300; rf_depth: int | None = None; rf_min_leaf: int = 1
    et_n_est: int = 300; et_depth: int | None = None
    cb_iterations: int = 300; cb_depth: int = 4; cb_lr: float = 0.05; cb_early_stop: int = 20
    gb_n_est: int = 200; gb_depth: int = 3; gb_lr: float = 0.05
    dl_hidden: tuple = (512, 256, 128)
    dl_dropout: tuple = (0.3, 0.3, 0.2)
    dl_lr: float = 1e-3; dl_epochs: int = 100; dl_batch: int = 32
    dl_early_stop: int = 15; dl_weight_decay: float = 1e-4

    # soft-vote weights: SVM XGB LGBM LR RF ET CB GB MLP
    w_svm: float = 1.0; w_xgb: float = 1.0; w_lgbm: float = 1.0; w_lr: float = 1.0
    w_rf: float = 1.0; w_et: float = 1.0; w_cb: float = 1.0; w_gb: float = 1.0; w_mlp: float = 1.0

    # SFS checkpoint files (same names as the earlier script)
    test_per_k: bool = True      # also log test metrics per k (reporting only; k is chosen from OOF AUC)
    train_csv: str = "VotingClassifier_v3_trainver3.csv"
    test_csv: str = "VotingClassifier_v3_testver3.csv"

    # reporting
    cons_thr: float = 0.60
    boot_B: int = 2000
    opt_seeds: tuple = (0, 1, 2)
    n_random_w: int = 200
    var_grid: tuple = (0.0, 0.05, 0.10, 0.25, 0.50)
    padj_grid: tuple = (0.001, 0.01, 0.05, 0.10)

    # switches for the slow parts
    run_ablation: bool = True
    run_shap: bool = True
    run_optimizer: bool = True
    run_stability: bool = True
    run_timing: bool = True
    run_sensitivity: bool = True

    data_dir: str = (r"C:\Users\chaulh\Downloads\Trang_tre_trau\paper\minh\paper 2026"
                     r"\Gene Expression Omnibus (GEO) Dataset GSE68086"
                     r"\python_desque2_lung\Binary\data")
    out_dir: str = (r"C:\Users\chaulh\Downloads\Trang_tre_trau\paper\minh\paper 2026"
                    r"\Gene Expression Omnibus (GEO) Dataset GSE68086"
                    r"\python_desque2_lung\Binary\data\final")


CFG = Config()
CFG.out_dir = os.path.join(CFG.data_dir, "TEP_VOTE_PYDESEQ2_CV")
REP_DIR = os.path.join(CFG.out_dir, "report")
MODEL_NAMES = ["SVM", "XGB", "LGBM", "LR", "RF", "ET", "CB", "GB", "MLP"]
WEIGHTS = [CFG.w_svm, CFG.w_xgb, CFG.w_lgbm, CFG.w_lr, CFG.w_rf, CFG.w_et, CFG.w_cb, CFG.w_gb, CFG.w_mlp]
DISPLAY = {"SVM": "SVM", "XGB": "XGBoost", "LGBM": "LightGBM", "LR": "Logistic Regression",
           "RF": "Random Forest", "ET": "Extra Trees", "CB": "CatBoost", "GB": "Gradient Boosting",
           "MLP": "Deep MLP", "LDA": "LDA", "ANN": "ANN", "TEP-VOTE": "TEP-VOTE"}
TABLE_ORDER = ["MLP", "XGB", "SVM", "CB", "LGBM", "GB", "LR", "ET", "RF", "LDA", "ANN", "TEP-VOTE"]
METRIC_COLS = ["Acc", "Sens", "Spec", "Prec", "F1", "AUC"]
ABL = {"imbalance": True}                     # toggled only by the ablation study
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log = logging.getLogger("tepvote")


def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ══════════════════════════════════════════════════════════════════════════
# [2] NORMALISATION + DESEQ2 + VST + FEATURE REPRESENTATION  (all fitted on a training split)
# ══════════════════════════════════════════════════════════════════════════
def fit_norm(train_counts: pd.DataFrame):
    """Zero filter + geometric means (training only).
    Geometric mean uses positive counts only (DESeq2 'poscounts' style) because classic
    median-of-ratios needs genes that are non-zero in every sample, which sparse data lacks."""
    keep = train_counts.columns[(train_counts == 0).mean(axis=0) <= CFG.zero_frac]
    c = train_counts[keep]
    log_gm = np.log(c.where(c > 0)).mean(axis=0, skipna=True)
    return {"cols": list(keep), "log_gm": log_gm}


def vst_closed_form(q, a0, a1):
    """DESeq2 parametric VST. q = size-factor-normalised counts; dispersion trend = a0 + a1 / mean."""
    return np.log2((1.0 + a1 + 2.0 * a0 * q + 2.0 * np.sqrt(a0 * q * (1.0 + a1 + a0 * q))) / (4.0 * a0))


def apply_vst(counts: pd.DataFrame, norm) -> pd.DataFrame:
    """Median-of-ratios size factors with TRAINING geometric means, then VST with the TRAINING
    dispersion-trend coefficients (norm['trend'] = (a0, a1)). Nothing is estimated from `counts`
    except each sample's own size factor."""
    c = counts[norm["cols"]]
    lr = np.log(c.where(c > 0)).sub(norm["log_gm"], axis=1)
    sf = np.exp(lr.median(axis=1, skipna=True))
    q = c.div(sf, axis=0).to_numpy(float)
    a0, a1 = norm["trend"]
    return pd.DataFrame(vst_closed_form(q, a0, a1), index=c.index, columns=c.columns)


def _trend_coeffs(dds):
    """Parametric dispersion-trend coefficients (a0, a1) stored by PyDESeq2 after dds.deseq2()."""
    tc = getattr(dds, "uns", {}).get("trend_coeffs", None)
    if tc is None:
        raise RuntimeError("PyDESeq2 did not store uns['trend_coeffs'] (dispersion trend not 'parametric', "
                           "or an incompatible pydeseq2 version). Try: pip install -U pydeseq2")
    try:
        a0, a1 = float(tc["a0"]), float(tc["a1"])
    except Exception:
        arr = np.asarray(tc, float).ravel()
        a0, a1 = float(arr[0]), float(arr[1])
    if not (np.isfinite(a0) and np.isfinite(a1) and a0 > 0):
        raise RuntimeError(f"Invalid dispersion-trend coefficients a0={a0}, a1={a1}")
    return a0, a1


def run_deseq2(counts_df: pd.DataFrame, labels: np.ndarray):
    """Returns (results DataFrame, (a0, a1)). Both come from the SAME training split."""
    for n in ["pydeseq2", "pydeseq2.dds", "pydeseq2.ds"]:
        logging.getLogger(n).setLevel(logging.WARNING)
    counts = counts_df.astype(int).copy()
    counts.index = [f"s{i}" for i in range(len(counts))]
    counts = counts.loc[:, counts.sum(axis=0) > 0]
    meta = pd.DataFrame({"condition": pd.Categorical(np.where(labels == 1, "cancer", "control"),
                                                      categories=["control", "cancer"])},
                        index=counts.index)
    dds = DeseqDataSet(counts=counts, metadata=meta, design="~ condition", refit_cooks=True, quiet=True)
    dds.deseq2()
    a0, a1 = _trend_coeffs(dds)
    log.info(f"dispersion trend: a0={a0:.5f}, a1={a1:.5f}")
    st = DeseqStats(dds, contrast=["condition", "cancer", "control"], quiet=True)
    st.summary()
    return st.results_df.dropna(subset=["padj", "log2FoldChange"]), (a0, a1)


def select_deg(res, padj=None, lfc=None, max_genes="cfg"):
    padj = CFG.deg_padj if padj is None else padj
    lfc = CFG.deg_lfc if lfc is None else lfc
    max_genes = CFG.deg_max if max_genes == "cfg" else max_genes
    sig = res[(res["padj"] < padj) & (res["log2FoldChange"].abs() > lfc)].sort_values("padj")
    g = sig.index.tolist()
    return g[:max_genes] if max_genes else g


def build_representation(train_counts, y, other_counts, use_deg=True, rank_by="anova", deg=None):
    """Everything fitted on `train_counts` only; `other_counts` are transformed with train parameters.
    `deg` = (results_df, (a0, a1)) can be passed to reuse an existing DESeq2 fit on the same training data.
    Returned matrices are columns-sorted (best gene first) and scaled, so top-k = first k columns."""
    norm = fit_norm(train_counts)
    if deg is None:
        deg = run_deseq2(train_counts[norm["cols"]].reset_index(drop=True), y)
    res, trend = deg
    norm["trend"] = trend
    Ltr = apply_vst(train_counts, norm)
    Lo = [apply_vst(c, norm) for c in other_counts]
    if use_deg:
        genes = [g for g in select_deg(res) if g in Ltr.columns]
    else:
        genes = list(Ltr.columns)
    var = Ltr[genes].var(ddof=1)
    genes = [g for g in genes if var[g] >= CFG.var_min]
    if not genes:
        raise RuntimeError("No genes left after DEG + variance filter")
    Xg = Ltr[genes].to_numpy(float)
    F, _ = f_classif(Xg, y)
    F = np.nan_to_num(F)
    order = np.argsort(-F, kind="stable") if rank_by == "anova" else np.arange(len(genes))
    genes = [genes[i] for i in order]
    sc = StandardScaler().fit(Xg[:, order])
    return {"genes": genes, "Xtr": sc.transform(Xg[:, order]),
            "Xo": [sc.transform(L[genes].to_numpy(float)) for L in Lo],
            "scaler": sc, "norm": norm, "res": res, "trend": trend, "F": pd.Series(F[order], index=genes)}


# ══════════════════════════════════════════════════════════════════════════
# [3] METRICS + MODEL FACTORIES
# ══════════════════════════════════════════════════════════════════════════
def soft_vote(probs, weights):
    w = np.asarray(weights, float); w = w / w.sum()
    return sum(p * wi for p, wi in zip(probs, w))


def youden_threshold(y, p) -> float:
    fpr, tpr, thr = roc_curve(y, p)
    ok = np.isfinite(thr)
    return float(thr[ok][np.argmax((tpr - fpr)[ok])])


def lofo_thresholds(y, p, folds_idx):
    """Leave-one-fold-out Youden threshold: the threshold applied to outer fold f is fitted on the
    OOF predictions of the other folds, so fold f's labels never choose its own threshold.
    Returns (per-sample threshold vector, list of per-fold thresholds)."""
    vec, per = np.zeros(len(y)), []
    for f, (_, va) in enumerate(folds_idx):
        others = np.concatenate([folds_idx[j][1] for j in range(len(folds_idx)) if j != f])
        t = youden_threshold(y[others], p[others])
        vec[va] = t; per.append(t)
    return vec, per


def metrics_at(y, p, thr):
    yhat = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) else 0.0
    spec = tn / (tn + fp) if (tn + fp) else 0.0
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    return {"Acc": 100 * (tp + tn) / (tp + tn + fp + fn), "Sens": 100 * sens, "Spec": 100 * spec,
            "Prec": 100 * prec, "F1": 100 * 2 * tp / max(2 * tp + fp + fn, 1),
            "AUC": 100 * roc_auc_score(y, p), "BalAcc": 50 * (sens + spec),
            "TP": tp, "TN": tn, "FP": fp, "FN": fn, "thr": thr}


def _smote(X, y):
    if not ABL["imbalance"]:
        return X, y
    return SMOTE(k_neighbors=CFG.smote_k, random_state=CFG.seed).fit_resample(X, y)


def _split(X, y, frac=0.15):
    return train_test_split(X, y, test_size=frac, stratify=y, random_state=CFG.seed)


def _smote_split(X, y):
    Xs, Xv, ys, yv = _split(X, y)
    Xs, ys = _smote(Xs, ys)
    return Xs, ys, Xv, yv


def fit_svm(X, y):
    Xs, ys = _smote(X, y)
    return SVC(kernel="rbf", C=CFG.svm_C, gamma=CFG.svm_gamma, probability=True,
               random_state=CFG.seed).fit(Xs, ys)


def fit_xgb(X, y):
    Xs, Xv, ys, yv = _split(X, y)
    spw = float((ys == 0).sum() / max((ys == 1).sum(), 1)) if ABL["imbalance"] else 1.0
    m = XGBClassifier(n_estimators=CFG.xgb_n_est, max_depth=CFG.xgb_depth, learning_rate=CFG.xgb_lr,
                      scale_pos_weight=spw, eval_metric="logloss",
                      early_stopping_rounds=CFG.xgb_early_stop, random_state=CFG.seed, n_jobs=-1)
    return m.fit(Xs, ys, eval_set=[(Xv, yv)], verbose=False)


def fit_lgbm(X, y):
    Xs, Xv, ys, yv = _split(X, y)
    m = LGBMClassifier(n_estimators=CFG.lgbm_n_est, max_depth=CFG.lgbm_depth, learning_rate=CFG.lgbm_lr,
                       is_unbalance=ABL["imbalance"], random_state=CFG.seed, n_jobs=-1, verbose=-1)
    return m.fit(Xs, ys, eval_set=[(Xv, yv)],
                 callbacks=[lgb.early_stopping(CFG.lgbm_early_stop, verbose=False), lgb.log_evaluation(-1)])


def fit_lr(X, y):
    Xs, ys = _smote(X, y)
    return LogisticRegression(solver="liblinear", penalty="l2", C=CFG.lr_C, max_iter=CFG.lr_max_iter,
                              random_state=CFG.seed).fit(Xs, ys)


def fit_rf(X, y):
    Xs, ys = _smote(X, y)
    return RandomForestClassifier(n_estimators=CFG.rf_n_est, max_depth=CFG.rf_depth,
                                  min_samples_leaf=CFG.rf_min_leaf, random_state=CFG.seed,
                                  n_jobs=-1).fit(Xs, ys)


def fit_et(X, y):
    Xs, ys = _smote(X, y)
    return ExtraTreesClassifier(n_estimators=CFG.et_n_est, max_depth=CFG.et_depth,
                                random_state=CFG.seed, n_jobs=-1).fit(Xs, ys)


def fit_cb(X, y):
    Xs, Xv, ys, yv = _split(X, y)
    m = CatBoostClassifier(iterations=CFG.cb_iterations, depth=CFG.cb_depth, learning_rate=CFG.cb_lr,
                           auto_class_weights="Balanced" if ABL["imbalance"] else None,
                           eval_metric="AUC", early_stopping_rounds=CFG.cb_early_stop,
                           random_seed=CFG.seed, verbose=False, thread_count=-1)
    return m.fit(Xs, ys, eval_set=(Xv, yv))


def fit_gb(X, y):
    Xs, ys = _smote(X, y)
    return GradientBoostingClassifier(n_estimators=CFG.gb_n_est, max_depth=CFG.gb_depth,
                                      learning_rate=CFG.gb_lr, random_state=CFG.seed).fit(Xs, ys)


class _MLP(nn.Module):
    def __init__(self, in_dim, hidden, dropout):
        super().__init__()
        layers, prev = [], in_dim
        for h, dr in zip(hidden, dropout):
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dr)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return torch.sigmoid(self.net(x)).squeeze(1)


def _adam(params):   # paper: Adam with L2 = 1e-4
    return torch.optim.Adam(params, lr=CFG.dl_lr, weight_decay=CFG.dl_weight_decay)


class MLPWrapper:
    def __init__(self, in_dim):
        self.in_dim, self.model = in_dim, None

    def fit(self, X, y, Xv_np, yv, pos_weight, make_opt=None, seed=None):
        torch.manual_seed(CFG.seed if seed is None else seed)
        self.model = _MLP(self.in_dim, CFG.dl_hidden, CFG.dl_dropout).to(DEVICE)
        opt = (make_opt or _adam)(self.model.parameters())
        crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], dtype=torch.float32, device=DEVICE))
        Xt = torch.tensor(X, dtype=torch.float32, device=DEVICE)
        yt = torch.tensor(y, dtype=torch.float32, device=DEVICE)
        Xv = torch.tensor(Xv_np, dtype=torch.float32, device=DEVICE)
        loader = DataLoader(TensorDataset(Xt, yt), batch_size=CFG.dl_batch, shuffle=True)
        best, pat, state = -float("inf"), 0, None
        for _ in range(CFG.dl_epochs):
            self.model.train()
            for xb, yb in loader:
                if len(xb) < 2:
                    continue
                opt.zero_grad()
                crit(self.model.net(xb).squeeze(1), yb).backward()
                opt.step()
            self.model.eval()
            with torch.no_grad():
                vp = self.model(Xv).cpu().numpy()
            try:
                va = roc_auc_score(yv, vp)
            except Exception:
                va = 0.0
            if va > best:
                best, pat = va, 0
                state = {k: v.detach().cpu().clone() for k, v in self.model.state_dict().items()}
            else:
                pat += 1
                if pat >= CFG.dl_early_stop:
                    break
        if state is None:
            raise RuntimeError("MLP checkpoint was not created")
        self.model.load_state_dict(state)
        return self

    def predict_proba(self, X):
        self.model.eval()
        with torch.no_grad():
            p = self.model(torch.tensor(X, dtype=torch.float32, device=DEVICE)).cpu().numpy()
        return np.column_stack([1 - p, p])


def fit_mlp(X, y, make_opt=None, seed=None):
    Xs, ys, Xv, yv = _smote_split(X, y)
    pw = float((ys == 0).sum() / max((ys == 1).sum(), 1)) if ABL["imbalance"] else 1.0
    return MLPWrapper(X.shape[1]).fit(Xs, ys, Xv, yv, pw, make_opt, seed)


def fit_lda(X, y):
    Xs, ys = _smote(X, y)
    return LinearDiscriminantAnalysis().fit(Xs, ys)


def fit_ann(X, y, solver="adam", seed=None):
    Xs, ys = _smote(X, y)
    return MLPClassifier(hidden_layer_sizes=(256, 128), activation="relu", solver=solver, alpha=1e-3,
                         batch_size=64, learning_rate_init=0.01, max_iter=800,
                         random_state=CFG.seed if seed is None else seed).fit(Xs, ys)


FIT_FNS = {"SVM": fit_svm, "XGB": fit_xgb, "LGBM": fit_lgbm, "LR": fit_lr, "RF": fit_rf,
           "ET": fit_et, "CB": fit_cb, "GB": fit_gb, "MLP": fit_mlp}
FIT_ALL = {**FIT_FNS, "LDA": fit_lda, "ANN": fit_ann}


def fit_all(X, y):
    return {n: FIT_FNS[n](X, y) for n in MODEL_NAMES}


def oof_nested(reps, y, folds_idx, k, fit_fns=None):
    """OOF probabilities where each fold uses ITS OWN gene list / normalisation, at a FIXED k."""
    fit_fns = fit_fns or FIT_FNS
    oof = {n: np.full(len(y), np.nan) for n in fit_fns}
    for f, (tr, va) in enumerate(folds_idx, 1):
        r = reps[f]; kk = min(k, r["Xtr"].shape[1])
        for n, fn in fit_fns.items():
            oof[n][va] = fn(r["Xtr"][:, :kk], y[tr]).predict_proba(r["Xo"][0][:, :kk])[:, 1]
    return oof


def oof_fixed(X, y, folds_idx, fit_fns=None):
    """OOF on one fixed matrix (used for ablations, SHAP, optimizer study)."""
    fit_fns = fit_fns or FIT_FNS
    oof = {n: np.full(len(y), np.nan) for n in fit_fns}
    for tr, va in folds_idx:
        for n, fn in fit_fns.items():
            oof[n][va] = fn(X[tr], y[tr]).predict_proba(X[va])[:, 1]
    return oof


# ── nested-k: k is selected INSIDE each outer training fold ─────────────
def inner_select_k(y_o, splits, k_grid, models):
    """splits = [(Xa, Xb, ia, ib), ...] inner train/val matrices (columns already ANOVA-ordered).
    Returns (k_star, {k: inner pooled AUC}). Ties go to the smaller k."""
    scores = {}
    for k in k_grid:
        oofk = {n: np.full(len(y_o), np.nan) for n in models}
        for Xa, Xb, ia, ib in splits:
            kk = min(k, Xa.shape[1])
            for n in models:
                oofk[n][ib] = FIT_FNS[n](Xa[:, :kk], y_o[ia]).predict_proba(Xb[:, :kk])[:, 1]
        p = soft_vote([oofk[n] for n in models], [1.0] * len(models))
        scores[k] = roc_auc_score(y_o, p)
    top = max(scores.values())
    return min(k for k, v in scores.items() if v >= top - 1e-12), scores


def nested_k_cv(X_train, y, folds_idx, reps, k_grid, cache_path, sig):
    """Honest CV: for each outer fold, pick k by inner CV on that fold's training samples only,
    refit all 11 models with that k, predict the outer fold. Returns (oof dict, chosen-k DataFrame)."""
    key = (sig, "nk_v1", CFG.inner_splits, CFG.inner_k_stride, tuple(CFG.inner_models),
           CFG.inner_full, tuple(k_grid))
    if CFG.resume and os.path.exists(cache_path):
        c = pickle.load(open(cache_path, "rb"))
        if c.get("key") == key:
            log.info("Loaded cached nested-k CV")
            return c["oof"], c["chosen"]
    oof = {n: np.full(len(y), np.nan) for n in FIT_ALL}
    chosen = []
    for f, (tr, va) in enumerate(folds_idx, 1):
        t0 = time.time()
        r, y_o = reps[f], y[tr]
        inner = list(StratifiedKFold(CFG.inner_splits, shuffle=True, random_state=CFG.seed + f)
                     .split(np.zeros(len(tr)), y_o))
        splits = []
        for ia, ib in inner:
            if CFG.inner_full:       # full nesting: DESeq2 + ANOVA rebuilt on the inner-train samples
                ri = build_representation(X_train.iloc[tr[ia]], y_o[ia], [X_train.iloc[tr[ib]]])
                splits.append((ri["Xtr"], ri["Xo"][0], ia, ib))
            else:                    # k-only nesting: outer fold's gene ranking is reused (see note in README)
                splits.append((r["Xtr"][ia], r["Xtr"][ib], ia, ib))
        k_star, sc = inner_select_k(y_o, splits, k_grid, CFG.inner_models)
        kk = min(k_star, r["Xtr"].shape[1])
        for n, fn in FIT_ALL.items():
            oof[n][va] = fn(r["Xtr"][:, :kk], y_o).predict_proba(r["Xo"][0][:, :kk])[:, 1]
        chosen.append({"fold": f, "k_selected": k_star, "inner_auc": sc[k_star]})
        print(f"  outer fold {f}: inner CV picked k={k_star} (inner AUC {sc[k_star]:.4f}) [{time.time() - t0:.0f}s]")
    chosen = pd.DataFrame(chosen)
    pickle.dump({"key": key, "oof": oof, "chosen": chosen}, open(cache_path, "wb"))
    return oof, chosen


# ══════════════════════════════════════════════════════════════════════════
# [4] STATS: bootstrap CI + DeLong
# ══════════════════════════════════════════════════════════════════════════
def bootstrap_auc_ci(y_true, y_score, B=2000, seed=42):
    y_true, y_score = np.asarray(y_true), np.asarray(y_score)
    rng = np.random.default_rng(seed)
    pos, neg = np.flatnonzero(y_true == 1), np.flatnonzero(y_true == 0)
    aucs = []
    for _ in range(B):
        idx = np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))])
        aucs.append(roc_auc_score(y_true[idx], y_score[idx]))
    return np.percentile(aucs, [2.5, 97.5])


def delong_roc_test(y_true, p1, p2):
    y_true, p1, p2 = np.asarray(y_true), np.asarray(p1), np.asarray(p2)
    pos, neg = y_true == 1, y_true == 0
    m, n = pos.sum(), neg.sum()

    def comp(p):
        x, y = p[pos], p[neg]
        psi = (x[:, None] > y[None, :]).astype(float) + 0.5 * (x[:, None] == y[None, :])
        return psi.mean(), psi.mean(axis=1), psi.mean(axis=0)

    a1, v10_1, v01_1 = comp(p1)
    a2, v10_2, v01_2 = comp(p2)
    cov = np.cov(np.vstack([v10_1, v10_2]), ddof=1) / m + np.cov(np.vstack([v01_1, v01_2]), ddof=1) / n
    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    if var <= 0:
        return a1, a2, np.nan, np.nan
    z = (a1 - a2) / np.sqrt(var)
    return a1, a2, z, 2 * stats.norm.sf(abs(z))


# ══════════════════════════════════════════════════════════════════════════
# [5] SHAP helpers
# ══════════════════════════════════════════════════════════════════════════
TREE_MODELS, LINEAR_MODELS, KERNEL_MODELS = {"XGB", "LGBM", "RF", "ET", "CB", "GB"}, {"LR"}, {"SVM", "MLP"}


def _pos_shap(sv, nf):
    sv = sv.values if hasattr(sv, "values") else sv
    if isinstance(sv, list) and len(sv) == 2:
        sv = np.asarray(sv[1])
    else:
        sv = np.asarray(sv)
        if sv.ndim == 3 and sv.shape[0] == 2 and sv.shape[-1] == nf:
            sv = sv[1]
        elif sv.ndim == 3 and sv.shape[-1] == 2 and sv.shape[1] == nf:
            sv = sv[:, :, 1]
        elif not (sv.ndim == 2 and sv.shape[-1] == nf):
            raise ValueError(f"Unexpected SHAP shape {sv.shape}")
    return np.abs(sv.astype(float)).mean(axis=0)


def shap_importance(name, model, X_bg, X_eval, nf):
    if name in TREE_MODELS:
        return _pos_shap(shap.TreeExplainer(model).shap_values(X_eval), nf)
    if name in LINEAR_MODELS:
        masker = shap.maskers.Independent(X_bg, max_samples=100)
        return _pos_shap(shap.LinearExplainer(model, masker).shap_values(X_eval), nf)
    rng = np.random.default_rng(0)
    bg = shap.kmeans(X_bg[rng.choice(len(X_bg), min(40, len(X_bg)), replace=False)], 10)
    ev = X_eval[rng.choice(len(X_eval), min(60, len(X_eval)), replace=False)]
    ex = shap.KernelExplainer(lambda x: model.predict_proba(x)[:, 1], bg)
    sv = np.asarray(ex.shap_values(ev, nsamples=100, silent=True))
    if sv.ndim == 3:
        sv = sv[0]
    return np.abs(sv).mean(axis=0)


def cv_shap(X, y, names, folds_idx):
    """Returns (mean importance DataFrame, list of per-fold DataFrames)."""
    per_fold = []
    for k, (tr, va) in enumerate(tqdm(folds_idx, desc="SHAP folds")):
        d = {}
        for n in MODEL_NAMES:
            try:
                d[n] = shap_importance(n, FIT_FNS[n](X[tr], y[tr]), X[tr], X[va], X.shape[1])
            except Exception as e:
                print(f"[WARN] fold {k} {n}: {e} -> zeros")
                d[n] = np.zeros(X.shape[1])
        per_fold.append(pd.DataFrame(d, index=names))
    return sum(per_fold) / len(per_fold), per_fold


def norm_cols(df):
    out = df.copy()
    for c in out.columns:
        lo, hi = out[c].min(), out[c].max()
        out[c] = 0.0 if hi == lo else (out[c] - lo) / (hi - lo)
    return out


def rank01(df):
    return df.rank(method="average").sub(1).div(len(df) - 1)


def weight_by_auc(norm_imp, auc_per_model, floor=0.5):
    aucs = pd.Series(auc_per_model, dtype=float)
    rng_ = aucs.max() - aucs.min()
    scaled = pd.Series(1.0, index=aucs.index) if rng_ < 1e-9 else floor + (1 - floor) * (aucs - aucs.min()) / rng_
    w = norm_imp.copy()
    for col in w.columns:
        w[col] = w[col] * scaled.get(col, 1.0)
    return norm_cols(w)


def plot_consensus_heatmap(norm_imp, top_n=30, save_path=None,
                           title="Performance-Weighted Gene Consensus\n(Weights based on train CV-AUC)"):
    df = norm_imp.copy()
    if top_n is not None:
        df = df.loc[df.mean(axis=1).sort_values(ascending=False).head(top_n).index]
    plt.figure(figsize=(1.3 * df.shape[1] + 3, 0.35 * df.shape[0] + 2))
    sns.heatmap(df, annot=True, fmt=".2f", cmap="YlGnBu", cbar_kws={"label": "Normalized Importance"},
                linewidths=0.4, linecolor="white")
    plt.title(title); plt.xlabel("Model Name (Weighted by AUC)"); plt.ylabel("Gene Name"); plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close()
    return df


def plot_cm(ax, y_true, p, thr, title):
    """thr may be a scalar or a per-sample array (used for the leave-one-fold-out thresholds)."""
    cm = confusion_matrix(y_true, (p >= thr).astype(int), labels=[0, 1])
    row_pct = cm / cm.sum(axis=1, keepdims=True) * 100
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]}\n({row_pct[i, j]:.1f}%)", ha="center", va="center", fontsize=12,
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["Pred 0", "Pred 1"]); ax.set_yticklabels(["True 0", "True 1"]); ax.set_title(title)


def roc_pr_all(preds, y, title, stem):
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 6))
    cmap = plt.get_cmap("tab20")
    for i, n in enumerate(TABLE_ORDER):
        p, vote = preds[n], n == "TEP-VOTE"
        kw = dict(color="black" if vote else cmap(i), lw=3 if vote else 1.4, ls="-" if vote or i % 2 == 0 else "--")
        fpr, tpr, _ = roc_curve(y, p); a1.plot(fpr, tpr, label=f"{DISPLAY[n]} ({roc_auc_score(y, p):.3f})", **kw)
        pr, rc, _ = precision_recall_curve(y, p); a2.plot(rc, pr, label=f"{DISPLAY[n]} ({average_precision_score(y, p):.3f})", **kw)
    a1.plot([0, 1], [0, 1], color="gray", ls=":"); a2.axhline(float(np.mean(y)), color="gray", ls=":")
    a1.set(xlabel="False positive rate", ylabel="True positive rate", title=f"ROC: {title}")
    a2.set(xlabel="Recall", ylabel="Precision", title=f"Precision-recall: {title}")
    a1.legend(loc="lower right", fontsize=7); a2.legend(loc="lower left", fontsize=7)
    for a in (a1, a2):
        a.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(os.path.join(REP_DIR, f"{stem}.png"), dpi=300)
    fig.savefig(os.path.join(REP_DIR, f"{stem}.pdf")); plt.close(fig)


def latex_rows(df, cols):
    for name, r in df.iterrows():
        print(f"{name:22s} & " + " & ".join(f"{r[c]:.2f}" for c in cols) + " \\\\")


# ══════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════
def main():
    logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
    seed_everything(CFG.seed)
    os.makedirs(CFG.out_dir, exist_ok=True); os.makedirs(REP_DIR, exist_ok=True)
    log.info(f"PyTorch MLP device: {DEVICE}")

    # ── load ────────────────────────────────────────────────────────────
    rd = lambda f: pd.read_csv(os.path.join(CFG.data_dir, f), low_memory=False)
    X_train, X_test, y_train, y_test = rd("X_train_lung2.csv"), rd("X_test_lung2.csv"), rd("y_train_lung2.csv"), rd("y_test_lung2.csv")
    for df in (X_train, X_test):
        if str(df.columns[0]).startswith("Unnamed"):
            df.drop(columns=df.columns[0], inplace=True)
    assert X_train.columns.equals(X_test.columns) and X_train.columns.is_unique
    assert list(y_train.columns) == ["cancer_type"] and len(X_train) == len(y_train) and len(X_test) == len(y_test)
    y_tr = y_train["cancer_type"].to_numpy(int); y_te = y_test["cancer_type"].to_numpy(int)
    for n, f in [("train", X_train), ("test", X_test)]:
        if not np.isfinite(f.to_numpy(float)).all():
            raise ValueError(f"{n} contains NaN/inf")
    print("Shapes:", X_train.shape, X_test.shape, "| train labels", np.unique(y_tr, return_counts=True),
          "| test labels", np.unique(y_te, return_counts=True))

    # optional Ensembl -> symbol (cached; skipped silently if offline)
    ens = [c for c in X_train.columns if str(c).startswith("ENSG")]
    if ens:
        map_csv = os.path.join(CFG.out_dir, "ensembl_to_symbol.csv")
        try:
            if os.path.exists(map_csv):
                m = pd.read_csv(map_csv); id2sym = dict(zip(m["column"], m["symbol"]))
            else:
                import mygene
                strip = {c: str(c).split(".")[0] for c in ens}
                res = mygene.MyGeneInfo().querymany(sorted(set(strip.values())), scopes="ensembl.gene",
                                                    fields="symbol", species="human", verbose=False)
                sym = {}
                for r in res:
                    if not r.get("notfound") and r.get("symbol") and r["query"] not in sym:
                        sym[r["query"]] = r["symbol"]
                id2sym = {c: sym.get(strip[c], c) for c in ens}
                pd.DataFrame({"column": list(id2sym), "symbol": list(id2sym.values())}).to_csv(map_csv, index=False)
            cnt = pd.Series(id2sym).value_counts()
            id2sym = {k: (f"{v}_{k}" if cnt[v] > 1 else v) for k, v in id2sym.items()}
            X_train, X_test = X_train.rename(columns=id2sym), X_test.rename(columns=id2sym)
        except Exception as e:
            print(f"[WARN] Ensembl->symbol mapping skipped: {e}")

    folds_idx = list(StratifiedKFold(n_splits=CFG.n_splits, shuffle=True, random_state=CFG.seed).split(X_train, y_tr))

    # cache signature: any change here invalidates the checkpoints below ("vst_v1" retires every log2-era cache)
    sig = (CFG.seed, CFG.n_splits, CFG.zero_frac, CFG.var_min, CFG.deg_padj, CFG.deg_lfc, CFG.deg_max,
           len(y_tr), "vst_v1")

    # ── Phase 1: per-fold normalisation + PyDESeq2 + VST + ANOVA ordering (checkpointed) ──
    print("\n=== Phase 1: per-fold preprocessing (norm -> PyDESeq2 -> VST -> var filter -> ANOVA order) ===")
    prep_path = os.path.join(CFG.out_dir, "prep_cache.pkl")
    reps, rep, deg_rec = None, None, []
    if CFG.resume and os.path.exists(prep_path):
        c = pickle.load(open(prep_path, "rb"))
        if c.get("sig") == sig:
            reps, rep, deg_rec = c["reps"], c["rep"], c["deg_rec"]
            log.info("Loaded cached preprocessing (per-fold + full-training DESeq2/VST)")
    if reps is None:
        reps, deg_rec = {}, []
        for f, (tr, va) in enumerate(folds_idx, 1):
            t0 = time.time()
            reps[f] = build_representation(X_train.iloc[tr], y_tr[tr], [X_train.iloc[va]])
            n_deg = len(select_deg(reps[f]["res"]))
            deg_rec.append({"fold": f, "n_deg": n_deg, "n_after_var_filter": len(reps[f]["genes"]),
                            "vst_a0": reps[f]["trend"][0], "vst_a1": reps[f]["trend"][1],
                            "genes": ",".join(reps[f]["genes"])})
            print(f"  fold {f}: {n_deg} DEGs -> {len(reps[f]['genes'])} after variance filter ({time.time() - t0:.0f}s)")
        rep = build_representation(X_train, y_tr, [X_test])      # full-training fit, used by the final model
        pickle.dump({"sig": sig, "reps": reps, "rep": rep, "deg_rec": deg_rec}, open(prep_path, "wb"))
    pd.DataFrame(deg_rec).to_csv(os.path.join(CFG.out_dir, "deg_genes_per_fold.csv"), index=False)

    # ── Phase 2: k sweep with pooled OOF (cached) -> picks the FINAL k ───
    max_k = min(CFG.sfs_max, max(len(r["genes"]) for r in reps.values()))
    cands = set(range(max(1, CFG.sfs_start), max_k + 1, CFG.sfs_step)) | {max_k}
    if CFG.best_k_override:
        cands.add(int(CFG.best_k_override))
    cands = sorted(cands)
    train_out = os.path.join(CFG.out_dir, CFG.train_csv)
    test_out = os.path.join(CFG.out_dir, CFG.test_csv)
    cache_path = os.path.join(CFG.out_dir, "oof_cache.pkl")
    cv_cache, test_cache = {}, {}
    if CFG.resume and os.path.exists(cache_path):
        c = pickle.load(open(cache_path, "rb"))
        if c.get("sig") == sig:
            cv_cache, test_cache = c["oof"], c.get("test", {})
            log.info(f"Resuming SFS: {len(set(cands) & set(cv_cache))}/{len(cands)} k-values already done")

    def _fmt(m):
        return {"ACC": m["Acc"], "Recall": m["Sens"], "Precision": m["Prec"], "Sp": m["Spec"],
                "Sn": m["Sens"], "f1_Score": m["F1"], "AUC": m["AUC"]}

    def save_sweep():
        """Rewrite the SFS CSVs after every k.
        'Gene' lists the top-k genes of the full-training ranking; the pooled OOF itself uses each fold's own genes."""
        rtr, rte = [], []
        for k_ in sorted(cv_cache):
            oof_ = cv_cache[k_]
            vote_ = soft_vote([oof_[n] for n in MODEL_NAMES], WEIGHTS)
            thr_ = youden_threshold(y_tr, vote_)
            g_ = rep["genes"][:min(k_, len(rep["genes"]))]
            rtr.append({"Gene": g_, "n_features": k_,
                        **{a: f"{b:.2f}%" for a, b in _fmt(metrics_at(y_tr, vote_, thr_)).items()},
                        **{f"AUC_{n}": f"{100 * roc_auc_score(y_tr, oof_[n]):.2f}%" for n in MODEL_NAMES}})
            if k_ in test_cache:
                pt_ = test_cache[k_]
                pv_ = soft_vote([pt_[n] for n in MODEL_NAMES], WEIGHTS)
                rte.append({"Gene": g_, "n_features": k_,
                            **{a: f"{b:.2f}%" for a, b in _fmt(metrics_at(y_te, pv_, thr_)).items()},
                            **{f"AUC_{n}": f"{100 * roc_auc_score(y_te, pt_[n]):.2f}%" for n in MODEL_NAMES}})
        pd.DataFrame(rtr).to_csv(train_out, index=False)
        if rte:
            pd.DataFrame(rte).to_csv(test_out, index=False)

    print(f"\n=== Phase 2: k sweep over {len(cands)} values (max_k={max_k}) ===")
    for k in tqdm(cands, desc="SFS"):
        if k not in cv_cache:
            cv_cache[k] = oof_nested(reps, y_tr, folds_idx, k)
        if CFG.test_per_k and k not in test_cache:          # reporting only, never used to pick k
            kk = min(k, rep["Xtr"].shape[1])
            mods = fit_all(rep["Xtr"][:, :kk], y_tr)
            test_cache[k] = {n: mods[n].predict_proba(rep["Xo"][0][:, :kk])[:, 1] for n in MODEL_NAMES}
            del mods
        pickle.dump({"sig": sig, "oof": cv_cache, "test": test_cache}, open(cache_path, "wb"))
        save_sweep()                                        # rolling checkpoint after every k

    cv_rows, fold_rows = [], []
    for k in sorted(cv_cache):
        oof = cv_cache[k]; vote = soft_vote([oof[n] for n in MODEL_NAMES], WEIGHTS)
        thr = youden_threshold(y_tr, vote); m = metrics_at(y_tr, vote, thr)
        cv_rows.append({"n_features": k, "threshold_oof": thr, **{c: m[c] for c in METRIC_COLS},
                        **{f"AUC_{n}": roc_auc_score(y_tr, oof[n]) for n in MODEL_NAMES}})
        for f, (_, va) in enumerate(folds_idx, 1):
            fm = metrics_at(y_tr[va], vote[va], 0.5)
            fold_rows.append({"n_features": k, "fold": f, **{c: fm[c] for c in METRIC_COLS}})
    cv_df = pd.DataFrame(cv_rows); cv_df.to_csv(os.path.join(CFG.out_dir, "pydeseq2_anova_oof_cv.csv"), index=False)
    pd.DataFrame(fold_rows).to_csv(os.path.join(CFG.out_dir, "pydeseq2_anova_cv_per_fold.csv"), index=False)
    best_k = int(CFG.best_k_override) if CFG.best_k_override else int(
        cv_df.sort_values(["AUC", "n_features"], ascending=[False, True]).iloc[0]["n_features"])
    oof_best = cv_cache[best_k]
    p_tr = soft_vote([oof_best[n] for n in MODEL_NAMES], WEIGHTS)       # pooled OOF at the selected k
    best_thr = youden_threshold(y_tr, p_tr)                              # deployment threshold for the final model
    print(f"\nSelected k={best_k} | pooled OOF AUC at selected k={roc_auc_score(y_tr, p_tr):.4f} "
          f"(selection-biased, NOT the CV figure to report) | Youden thr={best_thr:.4f}")

    # ── Phase 3: nested-k CV = the honest CV estimate ───────────────────
    k_grid = sorted(set(cands[::max(1, CFG.inner_k_stride)]) | {cands[-1]})
    print(f"\n=== Phase 3: nested-k CV (inner {CFG.inner_splits}-fold over k in {k_grid}, "
          f"proxy models {list(CFG.inner_models)}, inner_full={CFG.inner_full}) ===")
    oof_nk, chosen_df = nested_k_cv(X_train, y_tr, folds_idx, reps, k_grid,
                                    os.path.join(CFG.out_dir, "nested_k_cache.pkl"), sig)
    oof_nk = dict(oof_nk)
    oof_nk["TEP-VOTE"] = soft_vote([oof_nk[n] for n in MODEL_NAMES], WEIGHTS)
    p_nk = oof_nk["TEP-VOTE"]
    thr_nk = {n: lofo_thresholds(y_tr, oof_nk[n], folds_idx)[0] for n in oof_nk}   # per-sample threshold vectors
    chosen_df.to_csv(os.path.join(CFG.out_dir, "nested_k_choices.csv"), index=False)
    print(chosen_df.to_string(index=False))
    print(f"k chosen per outer fold: mean={chosen_df['k_selected'].mean():.1f}, "
          f"SD={chosen_df['k_selected'].std(ddof=1):.1f}, range={chosen_df['k_selected'].min()}-{chosen_df['k_selected'].max()}")
    print(f"TEP-VOTE OOF AUC | pooled at best k (biased): {roc_auc_score(y_tr, p_tr):.4f} | "
          f"nested-k (honest): {roc_auc_score(y_tr, p_nk):.4f}")

    # ── Final: full-training representation, ONE evaluation on test ─────
    print("\n=== Final held-out evaluation ===")
    res_full = rep["res"]          # full-training representation was built (or loaded) before the sweep
    kf = min(best_k, len(rep["genes"]))
    Xf_tr, Xf_te = rep["Xtr"][:, :kf], rep["Xo"][0][:, :kf]
    feature_names = rep["genes"][:kf]
    pd.DataFrame({"Rank": range(1, kf + 1), "Selected_Gene": feature_names}).to_csv(
        os.path.join(CFG.out_dir, "anova_selected_genes.csv"), index=False)
    pd.DataFrame({"Gene": rep["F"].index, "F_score": rep["F"].values}).to_csv(
        os.path.join(CFG.out_dir, "anova_f_test_gene_scores.csv"), index=False)
    print(f"{len(select_deg(res_full))} DEGs on full training -> {len(rep['genes'])} after variance filter -> using top {kf}")

    # ── persist exactly what the final models saw (no refit needed to plot later) ──
    prep_dir = os.path.join(CFG.out_dir, "prepared"); os.makedirs(prep_dir, exist_ok=True)
    Ltr_all = apply_vst(X_train, rep["norm"])       # VST values, train-fitted parameters
    Lte_all = apply_vst(X_test, rep["norm"])

    def _save(arr, cols, index, y, name):
        df = pd.DataFrame(arr, columns=cols, index=index); df["label"] = y
        df.to_csv(os.path.join(prep_dir, name))

    # best-k subset: scaled (model input) and VST (unscaled, for heatmaps/boxplots)
    _save(Xf_tr, feature_names, X_train.index, y_tr, "X_train_best_scaled.csv")
    _save(Xf_te, feature_names, X_test.index,  y_te, "X_test_best_scaled.csv")
    _save(Ltr_all[feature_names].to_numpy(), feature_names, X_train.index, y_tr, "X_train_best_vst.csv")
    _save(Lte_all[feature_names].to_numpy(), feature_names, X_test.index,  y_te, "X_test_best_vst.csv")

    # all ranked genes, so ANY k can be sliced later without refitting
    np.savez_compressed(os.path.join(prep_dir, "ranked_full.npz"),
                        genes=np.array(rep["genes"]), best_k=kf,
                        Xtr_scaled=rep["Xtr"], Xte_scaled=rep["Xo"][0],
                        Xtr_vst=Ltr_all[rep["genes"]].to_numpy(),
                        Xte_vst=Lte_all[rep["genes"]].to_numpy(),
                        y_tr=y_tr, y_te=y_te)

    final_models = fit_all(Xf_tr, y_tr)
    p_test_models = {n: m.predict_proba(Xf_te)[:, 1] for n, m in final_models.items()}
    p_te = soft_vote([p_test_models[n] for n in MODEL_NAMES], WEIGHTS)
    tm = metrics_at(y_te, p_te, best_thr)
    print("Held-out confusion matrix:\n", confusion_matrix(y_te, (p_te >= best_thr).astype(int)))
    print({k: round(float(v), 2) for k, v in tm.items() if k in METRIC_COLS + ["BalAcc"]})
    pd.DataFrame({"y_true": y_te, "y_prob": p_te, "y_pred": (p_te >= best_thr).astype(int)}).to_csv(
        os.path.join(CFG.out_dir, "heldout_predictions.csv"), index=False)
    pd.DataFrame([{"n_features": kf, "threshold": best_thr, **{c: tm[c] for c in METRIC_COLS},
                   **{f"AUC_{n}": roc_auc_score(y_te, p_test_models[n]) for n in MODEL_NAMES}}]).to_csv(
        os.path.join(CFG.out_dir, "pydeseq2_anova_final_heldout.csv"), index=False)

    ens_dir = os.path.join(CFG.out_dir, "saved_ensemble"); os.makedirs(ens_dir, exist_ok=True)
    for n, mdl in final_models.items():
        if n == "MLP":
            torch.save({"state_dict": mdl.model.state_dict(), "in_dim": mdl.in_dim,
                        "hidden": CFG.dl_hidden, "dropout": CFG.dl_dropout}, os.path.join(ens_dir, "mlp_state.pt"))
        else:
            joblib.dump(mdl, os.path.join(ens_dir, f"{n.lower()}_model.pkl"))
    pickle.dump({"norm": rep["norm"], "genes_ranked": rep["genes"], "scaler": rep["scaler"], "k": kf,
                 "weights": WEIGHTS, "model_names": MODEL_NAMES, "threshold": best_thr},
                open(os.path.join(ens_dir, "ensemble_meta.pkl"), "wb"))

    # ══ REPORT A: main table (9 learners + LDA + ANN + TEP-VOTE) ═════════
    # pooled OOF at the final k is used ONLY to set each model's deployment threshold for the test set
    oof_pool, test_all = dict(oof_best), dict(p_test_models)
    base = oof_nested(reps, y_tr, folds_idx, best_k, {"LDA": fit_lda, "ANN": fit_ann})
    for n in ("LDA", "ANN"):
        oof_pool[n] = base[n]; test_all[n] = FIT_ALL[n](Xf_tr, y_tr).predict_proba(Xf_te)[:, 1]
    oof_pool["TEP-VOTE"], test_all["TEP-VOTE"] = p_tr, p_te
    thr_all = {n: youden_threshold(y_tr, oof_pool[n]) for n in oof_pool}
    main_rows = {DISPLAY[n]: metrics_at(y_te, test_all[n], thr_all[n]) for n in TABLE_ORDER}
    df_main = pd.DataFrame(main_rows).T
    df_main.to_csv(os.path.join(REP_DIR, "table_main_results.csv"))
    print(f"\n===== MAIN RESULTS (held-out, k={kf}, per-model pooled-OOF Youden threshold) =====")
    print(df_main[METRIC_COLS + ["BalAcc", "TP", "TN", "FP", "FN"]].astype(float).round(2).to_string())
    latex_rows(df_main.astype(float), METRIC_COLS)

    # ── best-k ROC/PR, confusion matrices, per-fold TEP-VOTE table (train side = nested-k OOF) ──
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.8))
    series = [(y_tr, p_nk, "Train (5-fold nested-k OOF)", "tab:blue", "--"), (y_te, p_te, "Test", "tab:orange", "-")]
    for y_, p_, lab, c_, ls in series:
        fpr, tpr, _ = roc_curve(y_, p_)
        ax1.plot(fpr, tpr, color=c_, ls=ls, lw=2, label=f"{lab} (AUC = {roc_auc_score(y_, p_):.3f})")
        pr_, rc_, _ = precision_recall_curve(y_, p_)
        ax2.plot(rc_, pr_, color=c_, ls=ls, lw=2, label=f"{lab} (AUCpr = {average_precision_score(y_, p_):.3f})")
    ax1.plot([0, 1], [0, 1], color="gray", ls=":", lw=1)
    ax1.set(xlabel="False positive rate", ylabel="True positive rate", title=f"ROC curve ({kf} genes)")
    ax2.axhline(float(np.mean(y_te)), color="gray", ls=":", lw=1, label="Test prevalence")
    ax2.set(xlabel="Recall", ylabel="Precision", title=f"Precision-recall curve ({kf} genes)")
    ax1.legend(loc="lower right"); ax2.legend(loc="lower left"); ax1.grid(alpha=0.3); ax2.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(CFG.out_dir, "best_n_roc_pr.png"), dpi=300)
    fig.savefig(os.path.join(CFG.out_dir, "best_n_roc_pr.pdf")); plt.close(fig)

    fig_cm, (c1, c2) = plt.subplots(1, 2, figsize=(10, 4.5))
    plot_cm(c1, y_tr, p_nk, thr_nk["TEP-VOTE"], "Train (5-fold nested-k OOF)")
    plot_cm(c2, y_te, p_te, best_thr, f"Test, {kf} genes")
    fig_cm.tight_layout()
    fig_cm.savefig(os.path.join(CFG.out_dir, "best_n_confusion_matrix.png"), dpi=300)
    fig_cm.savefig(os.path.join(CFG.out_dir, "best_n_confusion_matrix.pdf")); plt.close(fig_cm)

    rows_cv = []
    for fi, (_, va) in enumerate(folds_idx, 1):
        m_ = metrics_at(y_tr[va], p_nk[va], thr_nk["TEP-VOTE"][va[0]])
        rows_cv.append({"Fold": fi, "k_selected": int(chosen_df.loc[chosen_df["fold"] == fi, "k_selected"].iloc[0]),
                        "ACC": m_["Acc"] / 100, "Sensitivity": m_["Sens"] / 100,
                        "Specificity": m_["Spec"] / 100, "Precision": m_["Prec"] / 100, "F1": m_["F1"] / 100,
                        "BalAcc": m_["BalAcc"] / 100, "AUC": m_["AUC"] / 100,
                        "AUCpr": average_precision_score(y_tr[va], p_nk[va])})
    df_cvb = pd.DataFrame(rows_cv).set_index("Fold")
    cols_ms = [c for c in df_cvb.columns if c != "k_selected"]
    mu_, sd_ = df_cvb[cols_ms].mean(), df_cvb[cols_ms].std(ddof=1)
    df_cvb.to_csv(os.path.join(CFG.out_dir, "best_n_train_per_fold.csv"))
    pd.DataFrame({"Metric": cols_ms,
                  "Mean ± SD (%)": [f"{mu_[m] * 100:.2f} ± {sd_[m] * 100:.2f}" for m in cols_ms]}).to_csv(
        os.path.join(CFG.out_dir, "best_n_train_mean_sd.csv"), index=False)

    # ══ REPORT B: CV mean ± SD  (nested-k OOF, leave-one-fold-out thresholds) ═════
    cv_ms, cv_f = {}, {}
    for n in TABLE_ORDER:
        d = pd.DataFrame([{c: metrics_at(y_tr[va], oof_nk[n][va], thr_nk[n][va[0]])[c] for c in METRIC_COLS}
                          for _, va in folds_idx])
        cv_f[DISPLAY[n]] = d
        cv_ms[DISPLAY[n]] = {c: f"{d[c].mean():.2f} ± {d[c].std(ddof=1):.2f}" for c in METRIC_COLS}
    df_cv_ms = pd.DataFrame(cv_ms).T
    df_cv_ms.to_csv(os.path.join(REP_DIR, "table_cv_mean_sd.csv"))
    pd.concat(cv_f, names=["Model", "Fold"]).to_csv(os.path.join(REP_DIR, "table_cv_per_fold.csv"))
    print("\n===== NESTED-k 5-FOLD CV (k chosen inside each outer fold; mean ± SD, %) =====\n", df_cv_ms.to_string())

    # ══ REPORT C: DeLong + bootstrap CI ═══════════════════════════════
    dl = []
    for n in TABLE_ORDER:
        lo, hi = bootstrap_auc_ci(y_te, test_all[n], CFG.boot_B)
        if n == "TEP-VOTE":
            dl.append({"Model": DISPLAY[n], "AUC": roc_auc_score(y_te, test_all[n]), "CI_low": lo, "CI_high": hi}); continue
        a_v, a_m, z, pv = delong_roc_test(y_te, test_all["TEP-VOTE"], test_all[n])
        dl.append({"Model": DISPLAY[n], "AUC": a_m, "CI_low": lo, "CI_high": hi, "dAUC": a_v - a_m, "z": z, "p": pv})
    df_dl = pd.DataFrame(dl).set_index("Model"); df_dl.to_csv(os.path.join(REP_DIR, "table_delong.csv"))
    print("\n===== DELONG (TEP-VOTE minus model); Bonferroni alpha = %.4f =====" % (0.05 / 11))
    print(df_dl.round(4).to_string())

    # ══ REPORT D: ablations ═══════════════════════════════════════════
    if CFG.run_ablation:
        def run_variant(label, Xa, Xb, imbalance=True):
            log.info(f"Ablation: {label} ({Xa.shape[1]} features)")
            ABL["imbalance"] = imbalance
            try:
                oof = oof_fixed(Xa, y_tr, folds_idx)
                thr = youden_threshold(y_tr, soft_vote([oof[n] for n in MODEL_NAMES], WEIGHTS))
                mods = fit_all(Xa, y_tr)
                p = soft_vote([mods[n].predict_proba(Xb)[:, 1] for n in MODEL_NAMES], WEIGHTS)
            finally:
                ABL["imbalance"] = True
            m = metrics_at(y_te, p, thr); m["n_features"] = Xa.shape[1]
            return m

        abl = {}
        deg_full = (res_full, rep["trend"])            # reuse the full-training DESeq2 fit (needed for VST too)
        rep_nodeg = build_representation(X_train, y_tr, [X_test], use_deg=False, deg=deg_full)
        rep_nopadj = build_representation(X_train, y_tr, [X_test], rank_by="padj", deg=deg_full)
        k_nd, k_np = min(best_k, len(rep_nodeg["genes"])), min(best_k, len(rep_nopadj["genes"]))
        abl["Without DEG filtering"] = run_variant("no DEG", rep_nodeg["Xtr"][:, :k_nd], rep_nodeg["Xo"][0][:, :k_nd])
        abl["Without ANOVA F-test ranking"] = run_variant("no ANOVA (padj order)", rep_nopadj["Xtr"][:, :k_np], rep_nopadj["Xo"][0][:, :k_np])
        abl["Without rank-guided SFS"] = run_variant("no SFS (all DEGs)", rep["Xtr"], rep["Xo"][0])
        abl["Without imbalance handling"] = run_variant("no imbalance handling", Xf_tr, Xf_te, imbalance=False)
        abl["TEP-VOTE (full)"] = {**main_rows["TEP-VOTE"], "n_features": kf}
        df_abl = pd.DataFrame(abl).T; df_abl.to_csv(os.path.join(REP_DIR, "table_ablation.csv"))
        print("\n===== ABLATION (held-out). Variants use a fixed-matrix CV for their threshold =====")
        print(df_abl[["n_features", "Acc", "F1", "AUC", "Sens", "Spec"]].astype(float).round(2).to_string())
        latex_rows(df_abl.astype({"Acc": float, "F1": float, "AUC": float}), ["Acc", "F1", "AUC"])

    # ══ REPORT E: SHAP consensus (+ stability, threshold sensitivity) ═════
    raw_imp, shap_folds = None, None
    if CFG.run_shap and SHAP_OK:
        # NOTE: interpretation uses the final k genes (chosen on the full training set) with CV models, as in the paper
        raw_imp, shap_folds = cv_shap(Xf_tr, y_tr, feature_names, folds_idx)
        raw_imp = raw_imp[MODEL_NAMES]; raw_imp.to_csv(os.path.join(REP_DIR, "shap_raw_importance.csv"))
        # performance-weighted heatmaps (YlGnBu, weights = nested-k OOF AUC per model)
        auc_pm = {n: roc_auc_score(y_tr, oof_nk[n]) for n in MODEL_NAMES}
        weighted_imp = weight_by_auc(norm_cols(raw_imp), auc_pm)
        weighted_imp.to_csv(os.path.join(REP_DIR, "shap_weighted_importance.csv"))
        plot_consensus_heatmap(weighted_imp, 30, os.path.join(REP_DIR, "shap_consensus_heatmap_top30.png"))
        plot_consensus_heatmap(weighted_imp, 15, os.path.join(REP_DIR, "shap_consensus_heatmap_top15.png"))
        show_m = [m for m in ["XGB", "MLP", "RF", "SVM", "LR", "ET", "CB"] if m in weighted_imp.columns]
        plot_consensus_heatmap(weighted_imp[show_m], 15, os.path.join(REP_DIR, "shap_consensus_heatmap_subset.png"))
        R = rank01(raw_imp); keep = (R > CFG.cons_thr).all(axis=1)
        A = pd.Series({n: np.mean([roc_auc_score(y_tr[va], oof_nk[n][va]) for _, va in folds_idx]) for n in MODEL_NAMES})
        weighted = norm_cols(raw_imp).mul(A, axis=1); S = weighted.sum(axis=1)
        cons = weighted.loc[keep].assign(S_g=S[keep]).sort_values("S_g", ascending=False)
        cons.rename(columns=DISPLAY).to_csv(os.path.join(REP_DIR, "shap_consensus_genes.csv"))
        print(f"\nConsensus genes (R>{CFG.cons_thr} in all {len(MODEL_NAMES)} models): {int(keep.sum())}")
        print(cons.head(15).rename(columns=DISPLAY).round(2).to_string())
        if len(cons):
            top = cons.head(15).drop(columns="S_g").rename(columns=DISPLAY)
            plt.figure(figsize=(1.2 * top.shape[1] + 3, 0.42 * len(top) + 2))
            sns.heatmap(top, annot=True, fmt=".2f", cmap="YlGnBu", linewidths=0.4, linecolor="white",
                        cbar_kws={"label": "Normalized importance × CV AUC"})
            plt.title("Top 15 consensus genes (ordered by $S_g$)"); plt.tight_layout()
            plt.savefig(os.path.join(REP_DIR, "shap_consensus_top15.png"), dpi=300, bbox_inches="tight"); plt.close()
        ref = set(R.index[keep])
        rows = [{"Threshold": t, "n_consensus_genes": len(s := set(R.index[(R > t).all(axis=1)])),
                 f"Jaccard vs {CFG.cons_thr}": len(s & ref) / max(len(s | ref), 1), "kept": len(s & ref)}
                for t in [0.5, 0.6, 0.7, 0.8, 0.9]]
        pd.DataFrame(rows).set_index("Threshold").to_csv(os.path.join(REP_DIR, "sensitivity_consensus_threshold.csv"))
        print("\n===== CONSENSUS THRESHOLD SENSITIVITY =====\n", pd.DataFrame(rows).round(3).to_string(index=False))
    elif CFG.run_shap:
        log.warning("shap not installed -> SHAP sections skipped")

    # ══ Stability ═════════════════════════════════════════════════════
    if CFG.run_stability:
        pairs = list(itertools.combinations(range(len(folds_idx)), 2))
        jac = lambda a, b: len(a & b) / len(a | b) if (a | b) else np.nan
        if shap_folds is not None:
            ref_genes = raw_imp.index[(rank01(raw_imp) > CFG.cons_thr).all(axis=1)].tolist(); NR = max(len(ref_genes), 1)
            R_f = [rank01(d[MODEL_NAMES]) for d in shap_folds]; M_f = [r.mean(axis=1) for r in R_f]
            topN = [set(m.nlargest(NR).index) for m in M_f]
            strict = [set(r.index[(r > CFG.cons_thr).all(axis=1)]) for r in R_f]
            rows = [{"Item": DISPLAY[n],
                     "Spearman": f"{np.mean(v := [spearmanr(shap_folds[i][n], shap_folds[j][n])[0] for i, j in pairs]):.3f} ± {np.std(v, ddof=1):.3f}",
                     f"Jaccard top-{NR}": f"{np.mean(w := [jac(set(shap_folds[i][n].nlargest(NR).index), set(shap_folds[j][n].nlargest(NR).index)) for i, j in pairs]):.3f} ± {np.std(w, ddof=1):.3f}"}
                    for n in MODEL_NAMES]
            v = [spearmanr(M_f[i], M_f[j])[0] for i, j in pairs]; w = [jac(topN[i], topN[j]) for i, j in pairs]
            rows.append({"Item": "Consensus score", "Spearman": f"{np.mean(v):.3f} ± {np.std(v, ddof=1):.3f}",
                         f"Jaccard top-{NR}": f"{np.mean(w):.3f} ± {np.std(w, ddof=1):.3f}"})
            df_st = pd.DataFrame(rows).set_index("Item"); df_st.to_csv(os.path.join(REP_DIR, "stability_shap_rank.csv"))
            print("\n===== SHAP STABILITY ACROSS FOLDS =====\n", df_st.to_string())
            rec = pd.DataFrame({f"in_fold_top{NR}": [sum(g in s for s in topN) for g in ref_genes],
                                "in_fold_strict": [sum(g in s for s in strict) for g in ref_genes]}, index=ref_genes)
            rec.to_csv(os.path.join(REP_DIR, "stability_gene_recurrence.csv"))
            print(f"Recurrence of {len(ref_genes)} consensus genes (5 folds):\n", rec.describe().loc[["mean"]].round(2).to_string())
        # ANOVA / selection consistency (each fold's own gene ranking)
        sel = [set(reps[f]["genes"][:min(best_k, len(reps[f]["genes"]))]) for f in range(1, len(folds_idx) + 1)]
        rho = []
        for i, j in pairs:
            common = reps[i + 1]["F"].index.intersection(reps[j + 1]["F"].index)
            if len(common) > 2:
                rho.append(spearmanr(reps[i + 1]["F"][common], reps[j + 1]["F"][common])[0])
        jf = [jac(sel[i], sel[j]) for i, j in pairs]
        print(f"\nANOVA consistency: F Spearman {np.mean(rho):.3f} ± {np.std(rho, ddof=1):.3f} | "
              f"top-{best_k} gene Jaccard {np.mean(jf):.3f} ± {np.std(jf, ddof=1):.3f}")
        pd.Series(feature_names).map(lambda g: sum(g in s for s in sel)).rename("n_folds_selected").to_frame().assign(
            gene=feature_names).to_csv(os.path.join(REP_DIR, "stability_anova_recurrence.csv"), index=False)

    # ══ Optimizer comparison ══════════════════════════════════════════
    if CFG.run_optimizer:
        OPTS = {"Adam (L2=1e-4) [used]": _adam,
                "AdamW (wd=1e-4)": lambda p: torch.optim.AdamW(p, lr=CFG.dl_lr, weight_decay=1e-4),
                "AdamW (wd=1e-2)": lambda p: torch.optim.AdamW(p, lr=CFG.dl_lr, weight_decay=1e-2),
                "SGD + momentum 0.9": lambda p: torch.optim.SGD(p, lr=1e-2, momentum=0.9, weight_decay=CFG.dl_weight_decay),
                "RMSprop": lambda p: torch.optim.RMSprop(p, lr=CFG.dl_lr, weight_decay=CFG.dl_weight_decay)}
        rows = []
        for oname, mk in OPTS.items():
            for sd in CFG.opt_seeds:
                log.info(f"Optimizer {oname} | seed {sd}")
                oof = np.zeros(len(y_tr))
                for tr, va in folds_idx:
                    oof[va] = fit_mlp(Xf_tr[tr], y_tr[tr], mk, sd).predict_proba(Xf_tr[va])[:, 1]
                pt = fit_mlp(Xf_tr, y_tr, mk, sd).predict_proba(Xf_te)[:, 1]
                rows.append({"Model": "MLP (PyTorch)", "Optimizer": oname, "Seed": sd, "CV_AUC": roc_auc_score(y_tr, oof),
                             "Test_AUC": roc_auc_score(y_te, pt), "Test_AUPRC": average_precision_score(y_te, pt)})
        for solver in ["adam", "sgd", "lbfgs"]:
            for sd in CFG.opt_seeds:
                oof = np.zeros(len(y_tr))
                for tr, va in folds_idx:
                    oof[va] = fit_ann(Xf_tr[tr], y_tr[tr], solver, sd).predict_proba(Xf_tr[va])[:, 1]
                pt = fit_ann(Xf_tr, y_tr, solver, sd).predict_proba(Xf_te)[:, 1]
                rows.append({"Model": "ANN (sklearn)", "Optimizer": solver, "Seed": sd, "CV_AUC": roc_auc_score(y_tr, oof),
                             "Test_AUC": roc_auc_score(y_te, pt), "Test_AUPRC": average_precision_score(y_te, pt)})
        raw = pd.DataFrame(rows); raw.to_csv(os.path.join(REP_DIR, "optimizer_raw.csv"), index=False)
        g = raw.groupby(["Model", "Optimizer"], sort=False)[["CV_AUC", "Test_AUC", "Test_AUPRC"]]
        df_opt = (100 * g.mean()).join(100 * g.std(ddof=1), rsuffix="_SD"); df_opt.to_csv(os.path.join(REP_DIR, "optimizer_summary.csv"))
        print("\n===== OPTIMIZER COMPARISON (%) =====\n", df_opt.round(2).to_string())

    # ══ ROC / PR all methods ══════════════════════════════════════════
    roc_pr_all(test_all, y_te, f"held-out test, k={kf}", "roc_pr_all_methods_test")
    roc_pr_all(oof_nk, y_tr, "nested-k OOF (k chosen per outer fold)", "roc_pr_all_methods_train_oof")

    # ══ Computational complexity ══════════════════════════════════════
    if CFG.run_timing:
        _ = fit_mlp(Xf_tr[:60], y_tr[:60])
        rows, ft, it = [], {}, {}
        for n in [m for m in TABLE_ORDER if m != "TEP-VOTE"]:
            a, b = [], []
            for _ in range(3):
                t0 = time.perf_counter(); mdl = FIT_ALL[n](Xf_tr, y_tr); a.append(time.perf_counter() - t0)
                t0 = time.perf_counter(); mdl.predict_proba(Xf_te); b.append(time.perf_counter() - t0)
            ft[n], it[n] = float(np.median(a)), float(np.median(b))
            rows.append({"Model": DISPLAY[n], "Train (s)": ft[n], "Inference (ms/sample)": 1000 * it[n] / len(Xf_te)})
        vf, vi = sum(ft[n] for n in MODEL_NAMES), sum(it[n] for n in MODEL_NAMES)
        rows.append({"Model": "TEP-VOTE (9 learners)", "Train (s)": vf, "Inference (ms/sample)": 1000 * vi / len(Xf_te)})
        df_cx = pd.DataFrame(rows).set_index("Model")
        df_cx["Train vs LDA (x)"] = df_cx["Train (s)"] / df_cx.loc["LDA", "Train (s)"]
        df_cx.to_csv(os.path.join(REP_DIR, "table_complexity.csv"))
        print(f"\n===== COMPLEXITY ({platform.processor() or platform.machine()}, {os.cpu_count()} CPUs, MLP on {DEVICE}) =====")
        print(df_cx.round(4).to_string())

    # ══ Sensitivity ═══════════════════════════════════════════════════
    if CFG.run_sensitivity:
        # K (pooled OOF AUC from the sweep; test AUC from a full-train refit at each K)
        Ks = sorted(k for k in cv_cache if k in (10, 25, 50, 75, 100, 125, 150, 175, 200, 250) or k == best_k)
        rows = []
        for k in Ks:
            if k in test_cache:
                pt = soft_vote([test_cache[k][n] for n in MODEL_NAMES], WEIGHTS)
            else:
                kk = min(k, rep["Xtr"].shape[1])
                mods = fit_all(rep["Xtr"][:, :kk], y_tr)
                pt = soft_vote([mods[n].predict_proba(rep["Xo"][0][:, :kk])[:, 1] for n in MODEL_NAMES], WEIGHTS)
            pv = soft_vote([cv_cache[k][n] for n in MODEL_NAMES], WEIGHTS)
            m = metrics_at(y_te, pt, youden_threshold(y_tr, pv))
            rows.append({"K": k, "Train CV AUC (%)": 100 * roc_auc_score(y_tr, pv), "Test AUC (%)": m["AUC"], "Test BalAcc (%)": m["BalAcc"]})
        df_K = pd.DataFrame(rows).set_index("K"); df_K.to_csv(os.path.join(REP_DIR, "sensitivity_K.csv"))
        print("\n===== SENSITIVITY: FEATURE COUNT K =====\n", df_K.round(2).to_string())
        fig, ax = plt.subplots(figsize=(7.5, 4.5))
        ax.plot(cv_df["n_features"], 100 * cv_df["AUC"] / 100, label="Pooled OOF (per k)"); ax.plot(df_K.index, df_K["Test AUC (%)"], "o-", label="Test")
        ax.axvline(best_k, color="gray", ls=":"); ax.set(xlabel="K", ylabel="AUC (%)"); ax.legend(); ax.grid(alpha=.3)
        fig.tight_layout(); fig.savefig(os.path.join(REP_DIR, "sensitivity_K.png"), dpi=300); plt.close(fig)

        # voting weights (no refit)
        oof_auc = pd.Series({n: roc_auc_score(y_tr, oof_best[n]) for n in MODEL_NAMES})

        def eval_w(w):
            po = soft_vote([oof_best[n] for n in MODEL_NAMES], list(w)); pt = soft_vote([p_test_models[n] for n in MODEL_NAMES], list(w))
            m = metrics_at(y_te, pt, youden_threshold(y_tr, po))
            return {"CV_AUC": 100 * roc_auc_score(y_tr, po), "Test_AUC": m["AUC"], "Test_AUPRC": 100 * average_precision_score(y_te, pt),
                    "F1": m["F1"], "Sens": m["Sens"], "Spec": m["Spec"]}

        schemes = {"Equal (current)": np.ones(9), "OOF-AUC weighted": oof_auc.values,
                   "(OOF-AUC - 0.5) weighted": (oof_auc - 0.5).clip(lower=1e-6).values,
                   "Top-3 by OOF AUC": oof_auc.rank(ascending=False).le(3).astype(float).values,
                   "Tree models only": np.array([n in TREE_MODELS for n in MODEL_NAMES], float)}
        for i, n in enumerate(MODEL_NAMES):
            w = np.ones(9); w[i] = 0; schemes[f"Drop {DISPLAY[n]}"] = w
        df_w = pd.DataFrame({k: eval_w(v) for k, v in schemes.items()}).T
        rng = np.random.default_rng(0)
        rnd = pd.DataFrame([eval_w(rng.dirichlet(np.ones(9))) for _ in range(CFG.n_random_w)])
        df_w.loc[f"Random Dirichlet (n={CFG.n_random_w}) mean"] = rnd.mean(); df_w.loc["Random Dirichlet SD"] = rnd.std(ddof=1)
        df_w.loc["Random Dirichlet min"] = rnd.min()
        df_w.to_csv(os.path.join(REP_DIR, "sensitivity_voting_weights.csv"))
        print("\n===== SENSITIVITY: VOTING WEIGHTS (%) =====\n", df_w.round(2).to_string())

        # variance cutoff + padj (re-built on the full-training representation, test AUC only)
        Ltr_f, Lte_f = apply_vst(X_train, rep["norm"]), apply_vst(X_test, rep["norm"])

        def run_setting(genes, label):
            genes = [g for g in genes if g in Ltr_f.columns]
            if len(genes) < 5:
                return None
            Xa, Xb = Ltr_f[genes].to_numpy(float), Lte_f[genes].to_numpy(float)
            F, _ = f_classif(Xa, y_tr); o = np.argsort(-np.nan_to_num(F), kind="stable")
            sc = StandardScaler().fit(Xa[:, o]); k_use = min(best_k, len(genes))
            Xa_s, Xb_s = sc.transform(Xa[:, o])[:, :k_use], sc.transform(Xb[:, o])[:, :k_use]
            mods = fit_all(Xa_s, y_tr)
            p = soft_vote([mods[n].predict_proba(Xb_s)[:, 1] for n in MODEL_NAMES], WEIGHTS)
            return {"Setting": label, "n_genes_in": len(genes), "n_genes_used": k_use,
                    "Test_AUC": 100 * roc_auc_score(y_te, p), "Test_AUPRC": 100 * average_precision_score(y_te, p)}

        deg_all = select_deg(res_full)
        var_f = Ltr_f.var(ddof=1)
        rows = [r for v in CFG.var_grid if (r := run_setting([g for g in deg_all if var_f[g] >= v], f"var >= {v}"))]
        df_var = pd.DataFrame(rows).set_index("Setting"); df_var.to_csv(os.path.join(REP_DIR, "sensitivity_variance_cutoff.csv"))
        print("\n===== SENSITIVITY: VARIANCE CUTOFF =====\n", df_var.round(2).to_string())
        rows = [r for t in CFG.padj_grid
                if (r := run_setting([g for g in select_deg(res_full, padj=t) if var_f.get(g, 0) >= CFG.var_min], f"padj < {t}"))]
        df_padj = pd.DataFrame(rows).set_index("Setting"); df_padj.to_csv(os.path.join(REP_DIR, "sensitivity_padj.csv"))
        print("\n===== SENSITIVITY: padj CUTOFF =====\n", df_padj.round(2).to_string())

    log.info(f"Done. Results in {CFG.out_dir}")


if __name__ == "__main__":     # required on Windows (PyDESeq2 / joblib spawn worker processes)
    main()

13:35:26 [INFO] PyTorch MLP device: cpu


Shapes: (728, 5440) (183, 5440) | train labels (array([0, 1]), array([312, 416])) | test labels (array([0, 1]), array([ 78, 105]))

=== Phase 1: per-fold preprocessing (norm -> PyDESeq2 -> VST -> var filter -> ANOVA order) ===


13:35:38 [INFO] dispersion trend: a0=0.43186, a1=9.79981


  fold 1: 217 DEGs -> 217 after variance filter (14s)


13:35:48 [INFO] dispersion trend: a0=0.42468, a1=9.35154


...

Recurrence of 23 consensus genes (5 folds):
       in_fold_top23  in_fold_strict
mean           3.65            2.26

ANOVA consistency: F Spearman 0.968 ± 0.009 | top-218 gene Jaccard 0.666 ± 0.048


15:13:13 [INFO] Optimizer Adam (L2=1e-4) [used] | seed 1
15:13:33 [INFO] Optimizer Adam (L2=1e-4) [used] | seed 2
15:13:59 [INFO] Optimizer AdamW (wd=1e-4) | seed 0
15:14:17 [INFO] Optimizer AdamW (wd=1e-4) | seed 1
15:14:38 [INFO] Optimizer AdamW (wd=1e-4) | seed 2
15:14:58 [INFO] Optimizer AdamW (wd=1e-2) | seed 0
15:15:13 [INFO] Optimizer AdamW (wd=1e-2) | seed 1
15:15:34 [INFO] Optimizer AdamW (wd=1e-2) | seed 2
15:15:51 [INFO] Optimizer SGD + momentum 0.9 | seed 0
15:16:05 [INFO] Optimizer SGD + momentum 0.9 | seed 1
15:16:25 [INFO] Optimizer SGD + momentum 0.9 | seed 2
15:16:40 [INFO] Optimizer RMSprop | seed 0
15:17:02 [INFO] Optimizer RMSprop | seed 1
15:17:24 [INFO] Optimizer RMSprop | seed 2



===== OPTIMIZER COMPARISON (%) =====
                                      CV_AUC  Test_AUC  Test_AUPRC  CV_AUC_SD  Test_AUC_SD  Test_AUPRC_SD
Model         Optimizer                                                                                 
MLP (PyTorch) Adam (L2=1e-4) [used]   95.10     94.30       96.28       0.56         0.35           0.06
              AdamW (wd=1e-4)         95.62     95.04       96.92       0.25         0.61           0.33
              AdamW (wd=1e-2)         95.50     94.64       96.56       0.42         0.85           0.54
              SGD + momentum 0.9      95.95     94.97       96.91       0.30         1.00           0.54
              RMSprop                 95.38     95.95       97.42       0.07         0.49           0.34
ANN (sklearn) adam                    96.33     94.61       96.63       0.30         0.22           0.14
              sgd                     96.72     95.12       96.95       0.01         0.22           0.11
              lb

15:22:07 [INFO] Done. Results in C:\Users\chaulh\Downloads\Trang_tre_trau\paper\minh\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python_desque2_lung\Binary\data\TEP_VOTE_PYDESEQ2_CV



===== SENSITIVITY: padj CUTOFF =====
               n_genes_in  n_genes_used  Test_AUC  Test_AUPRC
Setting                                                     
padj < 0.001         201           201     96.35       97.66
padj < 0.01          201           201     96.35       97.66
padj < 0.05          201           201     96.35       97.66
padj < 0.1           201           201     96.35       97.66


In [2]:
# ═══════════════════════════════════════════════════════════════════════
# CELL: Top-15 consensus genes -> DE table, per-gene ROC (95% CI), correlation,
#       group comparison, multivariable logistic regression (odds ratios)
# Needs: prep_cache.pkl, prepared/ranked_full.npz, report/shap_consensus_genes.csv
# ═══════════════════════════════════════════════════════════════════════
import importlib.util, subprocess, sys, os, pickle
if importlib.util.find_spec("statsmodels") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "statsmodels"])

import numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt, seaborn as sns
import statsmodels.api as sm
from scipy.stats import mannwhitneyu, spearmanr
from sklearn.metrics import roc_auc_score, roc_curve
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor

try:
    OUT = CFG.out_dir
except NameError:   # <- edit if running without the main script loaded
    OUT = (r"C:\Users\chaulh\Downloads\Trang_tre_trau\paper\minh\paper 2026"
           r"\Gene Expression Omnibus (GEO) Dataset GSE68086\python_desque2_lung\Binary\data\TEP_VOTE_PYDESEQ2_CV")
REP = os.path.join(OUT, "report"); os.makedirs(REP, exist_ok=True)
TOP_N, B = 15, 2000

# ── load what the pipeline already saved ───────────────────────────────
prep = pickle.load(open(os.path.join(OUT, "prep_cache.pkl"), "rb"))
res_full = prep["rep"]["res"]                                   # PyDESeq2 results, full training set
npz = np.load(os.path.join(OUT, "prepared", "ranked_full.npz"), allow_pickle=True)
genes_all = list(npz["genes"])
Vtr = pd.DataFrame(npz["Xtr_vst"], columns=genes_all)           # VST, train-fitted parameters
Vte = pd.DataFrame(npz["Xte_vst"], columns=genes_all)
y_tr, y_te = npz["y_tr"].astype(int), npz["y_te"].astype(int)

cons = pd.read_csv(os.path.join(REP, "shap_consensus_genes.csv"), index_col=0)
top = [g for g in cons.index if g in Vtr.columns][:TOP_N]
if len(top) < TOP_N:   # fallback: fewer consensus genes than 15 -> fill by mean weighted SHAP
    w = pd.read_csv(os.path.join(REP, "shap_weighted_importance.csv"), index_col=0).mean(axis=1)
    top += [g for g in w.sort_values(ascending=False).index if g not in top and g in Vtr.columns][:TOP_N - len(top)]
print(f"{len(top)} genes:", top)

rng = np.random.default_rng(42)
def boot_auc(y, s):
    pos, neg = np.flatnonzero(y == 1), np.flatnonzero(y == 0); a = []
    for _ in range(B):
        i = np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))])
        a.append(roc_auc_score(y[i], s[i]))
    return np.percentile(a, [2.5, 97.5])

def youden(y, s):
    f, t, th = roc_curve(y, s); ok = np.isfinite(th)
    return th[ok][np.argmax((t - f)[ok])]

def sens_spec(y, s, thr):
    p = s >= thr
    return 100 * (p & (y == 1)).sum() / (y == 1).sum(), 100 * (~p & (y == 0)).sum() / (y == 0).sum()

# ── 1) DE table + per-gene ROC + group comparison ──────────────────────
rows = []
for g in top:
    r = res_full.loc[g]
    sgn = 1 if r["log2FoldChange"] > 0 else -1                  # orient score so AUC >= 0.5 when direction matches
    s_tr, s_te = sgn * Vtr[g].to_numpy(), sgn * Vte[g].to_numpy()
    thr = youden(y_tr, s_tr)                                     # threshold from TRAIN only
    a_tr, a_te = roc_auc_score(y_tr, s_tr), roc_auc_score(y_te, s_te)
    lo_tr, hi_tr = boot_auc(y_tr, s_tr); lo_te, hi_te = boot_auc(y_te, s_te)
    se_te, sp_te = sens_spec(y_te, s_te, thr)
    rho, p_rho = spearmanr(Vte[g], y_te)
    p_mw = mannwhitneyu(Vte.loc[y_te == 1, g], Vte.loc[y_te == 0, g], alternative="two-sided").pvalue
    rows.append({"Gene": g, "Direction": "Up in NSCLC" if sgn > 0 else "Down in NSCLC",
                 "baseMean": r["baseMean"], "log2FC": r["log2FoldChange"], "lfcSE": r["lfcSE"],
                 "p_value": r["pvalue"], "padj_FDR": r["padj"],
                 "AUC_train": a_tr, "AUC_train_CI": f"{lo_tr:.3f}-{hi_tr:.3f}",
                 "AUC_test": a_te, "AUC_test_CI": f"{lo_te:.3f}-{hi_te:.3f}",
                 "Sens_test_%": se_te, "Spec_test_%": sp_te,
                 "Spearman_rho_test_vs_label": rho, "MannWhitney_p_test": p_mw})
de = pd.DataFrame(rows).set_index("Gene")
de["MannWhitney_FDR_test"] = multipletests(de["MannWhitney_p_test"], method="fdr_bh")[1]
de.to_csv(os.path.join(REP, "top15_DE_ROC_table.csv"))
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
print("\n===== TOP-15: DE + per-gene ROC (threshold from train, applied to test) =====")
print(de.round(4).to_string())

# ── 2) Correlation among the top-15 genes (train, Spearman) ────────────
C = Vtr[top].corr(method="spearman"); C.to_csv(os.path.join(REP, "top15_spearman_corr.csv"))
plt.figure(figsize=(9, 7.5))
sns.heatmap(C, cmap="RdBu_r", vmin=-1, vmax=1, annot=True, fmt=".2f", annot_kws={"size": 7},
            linewidths=.3, cbar_kws={"label": "Spearman rho"})
plt.title("Spearman correlation of the top-15 consensus genes (training VST)")
plt.tight_layout(); plt.savefig(os.path.join(REP, "top15_corr_heatmap.png"), dpi=300); plt.close()
iu = np.triu_indices(len(top), 1)
print(f"\nMedian |rho| between genes: {np.median(np.abs(C.values[iu])):.2f} | max |rho|: {np.abs(C.values[iu]).max():.2f}")

# ── 3) Multivariable logistic regression (per-SD odds ratios) ──────────
mu, sd = Vtr[top].mean(), Vtr[top].std(ddof=1)
Ztr, Zte = (Vtr[top] - mu) / sd, (Vte[top] - mu) / sd           # train mean/SD only
Xm = sm.add_constant(Ztr)
try:
    fit = sm.Logit(y_tr, Xm).fit(disp=0, maxiter=200)
    method = "unpenalised ML logistic regression"
except Exception as e:                                           # separation / singular -> ridge-penalised fallback
    print(f"[WARN] ML fit failed ({e}); using L2-penalised fit (CIs are approximate)")
    fit = sm.Logit(y_tr, Xm).fit_regularized(alpha=1e-2, disp=0); method = "L2-penalised (approx.)"
    fit.bse = getattr(fit, "bse", pd.Series(np.nan, index=Xm.columns))
ci = fit.conf_int()
orr = pd.DataFrame({"OR_per_SD": np.exp(fit.params), "CI_low": np.exp(ci[0]), "CI_high": np.exp(ci[1]),
                    "p_value": fit.pvalues}).drop(index="const")
orr["VIF"] = [variance_inflation_factor(Xm.values, i + 1) for i in range(len(top))]
orr["FDR"] = multipletests(orr["p_value"], method="fdr_bh")[1]
orr.to_csv(os.path.join(REP, "top15_multivariable_logistic.csv"))
p_lr_te = fit.predict(sm.add_constant(Zte, has_constant="add"))
print(f"\n===== MULTIVARIABLE LOGISTIC REGRESSION ({method}) =====")
print(orr.round(4).to_string())
print(f"15-gene logistic model: train AUC {roc_auc_score(y_tr, fit.predict(Xm)):.4f} | "
      f"test AUC {roc_auc_score(y_te, p_lr_te):.4f} (95% CI {boot_auc(y_te, np.asarray(p_lr_te))[0]:.3f}-"
      f"{boot_auc(y_te, np.asarray(p_lr_te))[1]:.3f})")

# forest plot
o = orr.sort_values("OR_per_SD"); fig, ax = plt.subplots(figsize=(6.5, 0.38 * len(o) + 1.8))
ax.errorbar(o["OR_per_SD"], range(len(o)), xerr=[o["OR_per_SD"] - o["CI_low"], o["CI_high"] - o["OR_per_SD"]],
            fmt="s", color="black", capsize=3)
ax.axvline(1, color="gray", ls="--"); ax.set_xscale("log"); ax.set_yticks(range(len(o))); ax.set_yticklabels(o.index)
ax.set(xlabel="Adjusted odds ratio per 1 SD (95% CI, log scale)", title="Multivariable logistic regression, top-15 genes")
ax.grid(alpha=.3); fig.tight_layout(); fig.savefig(os.path.join(REP, "top15_forest_OR.png"), dpi=300); plt.close(fig)
print(f"\nSaved to {REP}: top15_DE_ROC_table.csv, top15_spearman_corr.csv, top15_multivariable_logistic.csv, "
      f"top15_corr_heatmap.png, top15_forest_OR.png")

15 genes: ['IFITM3', 'WDR11-DT', 'HSD17B3', 'LINC01088', 'MALAT1', 'NOMO1', 'CRYM', 'COL6A3', 'MAOB', 'CA1', 'ZNF542P', 'ZNF385D', 'FCMR', 'ALAS2', 'IGFBP2']

===== TOP-15: DE + per-gene ROC (threshold from train, applied to test) =====
               Direction  baseMean  log2FC   lfcSE  p_value  padj_FDR  AUC_train AUC_train_CI  AUC_test  AUC_test_CI  Sens_test_%  Spec_test_%  Spearman_rho_test_vs_label  MannWhitney_p_test  MannWhitney_FDR_test
Gene                                                                                                                                                                                                                
IFITM3       Up in NSCLC   56.3528  1.5378  0.0889      0.0       0.0     0.8000  0.767-0.832    0.7846  0.716-0.851      68.5714      80.7692                      0.4876              0.0000                0.0000
WDR11-DT   Down in NSCLC   71.3204 -1.0790  0.0634      0.0       0.0     0.7837  0.748-0.818    0.7884  0.714-0.854      81